# Single-Sideband Modulation Demo

This notebook is a template for producing single-sideband modulated pulses using the RFSoC 4x2.

In [13]:
# Import libraries
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../rfsoc_lib')
from rfsoc_calibration import RfsocCalibration
from rfsoc_pulses import RfsocPulses

import pickle
%pylab inline

import rpyc
config = {'allow_public_attrs': True, 'allow_all_attrs': True, 'allow_setattr':True, 'allow_delattr': True, 'allow_pickle':True, "sync_request_timeout": 300}
exp = rpyc.connect("localhost", port=12346, config=config).root
import dill
obtain = lambda obj: dill.loads(dill.dumps(obj)) # Do obtain(obj) to transfer obj from the server to here

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


C:\ProgramData\miniconda3\Lib\site-packages\IPython\core\magics\pylab.py:162: UserWarning: pylab import has clobbered these variables: ['histogram', 'exp']
`%matplotlib` prevents importing * from pylab and numpy
  warn("pylab import has clobbered these variables: %s"  % clobbered +


In [14]:
# Board libraries
from qick import *
import Pyro4

# Pyro communicates with board
Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION=4

# Static IP proxy
ns_host = "172.24.106.21" # <-- SET IP ADDRESS
ns_port = 8888
proxy_name = "myqick"

# QICK object
ns = Pyro4.locateNS(host=ns_host, port=ns_port)
soc = Pyro4.Proxy(ns.lookup(proxy_name))
soccfg = QickConfig(soc.get_cfg())
print(soccfg)


QICK configuration:

	Board: RFSoC4x2

	Software version: 0.2.267
	Firmware timestamp: Wed Sep  6 18:49:29 2023

	Global clocks (MHz): tProcessor 409.600, RF reference 491.520

	2 signal generator channels:
	0:	axis_signal_gen_v6 - envelope memory 65536 samples (6.667 us)
		fs=9830.400 MHz, fabric=614.400 MHz, 32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - envelope memory 65536 samples (6.667 us)
		fs=9830.400 MHz, fabric=614.400 MHz, 32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 MHz, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		maxlen 16384 accumulated, 1024 decimated (1.852 us)
		triggered by output 7, pin 14, feedback to tProc input 0
		ADC tile 0, blk 0 is ADC_D
	1:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 MHz, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		maxlen 16384 accumulated, 1024 decimated (1.852 us)
		triggered by o

In [15]:
# Import dictionary of sequences from pickle file
# with open('pulse_sequences/Sequence_ch1_ch2_mw.pickle', 'rb') as handle:
#     imported_sequences = pickle.load(handle)

deg_to_rad = lambda deg: deg*np.pi/180
# Test pulse sequence
initial_delay = 0 
delay_raman = 442 + 2
delay_pump = 412 
delay_analog = 30 + delay_pump

delay_mw = 0
mw_freq_ghz = 0.5
buffer = 20

pulse_length_ns = 50
raman_cooling = 10000
ramsey_delay = 50
ramsey_phase  = deg_to_rad(45)

rf_pulse = [(initial_delay + delay_pump,0), (raman_cooling, 0),  (buffer, 0), (2500, 1), 
            (buffer, 0), (pulse_length_ns, 0), 
                (ramsey_delay, 0), 
                (pulse_length_ns, 0), (buffer, 0),
                (2500, 1), (500, 0)]

raman_pulse = [(initial_delay+ delay_raman, 0), (raman_cooling, 1),  (buffer, 0), (2500, 0), 
               (buffer, 1), (pulse_length_ns, 1), 
                (ramsey_delay, 1), 
                (pulse_length_ns, 1), (buffer,1),
                (2500, 0), (500, 0)]

analog_pulse = [(initial_delay+ delay_analog, 0.15), (raman_cooling, 0.15),  (buffer, 1), (2500, 1), 
                (buffer, 1), (pulse_length_ns, 1), 
                (ramsey_delay, 1), 
                (pulse_length_ns, 1), (buffer, 1),
                (2500, 1), (500, 1)]
# mw_seq = [(initial_delay + delay_mw,0), (raman_cooling+3*buffer+2*2500+pulse_length_ns, 1.0,mw_freq_ghz, 0), (500, 0)]
mw_seq = [(initial_delay + delay_mw,0), (raman_cooling, 1.0,mw_freq_ghz, 0),  (raman_cooling, 1.0,mw_freq_ghz, 0), (buffer, 0), (2500, 0), 
          (buffer, 0), (pulse_length_ns, 1.0, mw_freq_ghz, 0), 
          (ramsey_delay, 0), 
          (pulse_length_ns, 1.0, mw_freq_ghz, ramsey_phase), (buffer, 0),
          (2500, 0), (500, 0)]
imported_sequences = {}
imported_sequences["MW"] = mw_seq
seq_length =  int(np.sum(np.array(rf_pulse)[:,0]))
print (int(np.sum(np.array(rf_pulse)[:,0])), int(np.sum(np.array(raman_pulse)[:,0])),
      int(np.sum(np.array(analog_pulse)[:,0])))
# imported_sequences["DIG"] = [(1000, 1)]

16122 16154 16152


In [16]:
# configure trigger.
ps = exp.get_device('PulseStreamer') # get the pulse streamer
counter = exp.get_device('Counter')

seq = ps.device.createSequence() # create new sequence
seq.setDigital(0, rf_pulse)
seq.setDigital(1, raman_pulse)
seq.setDigital(6, [(10, 1), (int(seq_length-10), 0)]) # trigger the time tagger
seq.setDigital(2, [(10, 1), (int(seq_length-10), 0)]) # triggger the AWG/RFSoc
seq.setAnalog(0, analog_pulse)
ps.play_seq(seq) # start playback of the sequence


In [17]:
dac_delay = 0
# dac_gain = 20000

# Assign imported pulse sequence to DAC_A and DAC_B, setting gain and delays
pulses = RfsocPulses(imported_sequences,
                              ch_map = {"DAC_A":"MW", "DAC_B":"MW"},
                              delays = {"DAC_A":dac_delay, "DAC_B":dac_delay},
                            #   gains  = {"DAC_A":dac_gain, "DAC_B":dac_gain},
                              )

----- DAC A -----
ch_type: DAC
ch_index: 1
delay: 0
gain: 10000
lengths: [10.0, 10.0, 0.05, 0.05]
times: [0.0, 10.0, 22.54, 22.64]
amps: [1.0, 1.0, 1.0, 1.0]
freqs: [500.0, 500.0, 500.0, 500.0]
phases: [0.0, 0.0, 0.0, 45.0]
num_pulses: 4
duration: 25.71
----- DAC B -----
ch_type: DAC
ch_index: 0
delay: 0
gain: 10000
lengths: [10.0, 10.0, 0.05, 0.05]
times: [0.0, 10.0, 22.54, 22.64]
amps: [1.0, 1.0, 1.0, 1.0]
freqs: [500.0, 500.0, 500.0, 500.0]
phases: [0.0, 0.0, 0.0, 45.0]
num_pulses: 4
duration: 25.71
----- End time: 25.71 -----


In [18]:
# Import DAC and ssb calibrations
with open('../pickles/calibrate_phase.pickle', 'rb') as handle:
    dac_phis = pickle.load(handle)

with open('../pickles/ssb_params_30000.pickle', 'rb') as handle:
    ssb_params = pickle.load(handle)

with open('../pickles/const_power.pickle', 'rb') as handle:
    const_power = pickle.load(handle)

calibration = RfsocCalibration(dac_phis, ssb_params)

In [19]:
from IPython.display import display, clear_output
import time

# Program RFSoC with pulse sequence and configure for an external trigger (PMOD1_0)
internal = False
# Stop DACs
try:
    soc.reset_gens()
except Exception as ep:
    print (ep)
    pass

prog = QickProgram(soccfg)

if internal:
    print ('Internal generation')
    pulses.generate_asm(prog, calibration, const_power, const_power_factor=1, reps=100000)
    pulses.config_internal_start(soc, prog)
    soc.tproc.start()
else:
    print ('External generation')
    pulses.generate_asm(prog, calibration, const_power, const_power_factor=1, reps=1)
    pulses.config_external_start(soc, prog)

print(prog)

External generation

// Program

        synci 200;
        regwi 0, $14, 0;
LOOP_I: regwi 1, $22, 218453333;                //freq = 218453333
        regwi 1, $23, 0;                        //phase = 0
        regwi 1, $25, 20715;                    //gain = 20715
        regwi 1, $26, 595968;                   //phrst| stdysel | mode | | outsel = 0b01001 | length = 6144 
        regwi 1, $27, 0;                        //t = 0
        set 1, 1, $22, $23, $0, $25, $26, $27;  //ch = 1, pulse @t = $27
        regwi 1, $22, 218453333;                //freq = 218453333
        regwi 1, $23, 0;                        //phase = 0
        regwi 1, $25, 20715;                    //gain = 20715
        regwi 1, $26, 595968;                   //phrst| stdysel | mode | | outsel = 0b01001 | length = 6144 
        regwi 1, $27, 4096;                     //t = 4096
        set 1, 1, $22, $23, $0, $25, $26, $27;  //ch = 1, pulse @t = $27
        regwi 1, $22, 218453333;                //freq = 21845

In [20]:

polling_interval = 0
integration_time_s = 1
ch_trig = 6
ch_click = 4#1 #dd one to channel number of de~vice. # 2: SNSPD-ch1, 5: APD, 6: trigger mira
n_points = 50
bin_ns = 2
seq_length = seq_length

# from IPython.display import display, HTML
# display(HTML("<style>.container { widt~h:100% !important; }</style>"))
# plt.rcParams.update({'font.size': 40})


# plt.rcParams.update({'font.size': 40})

histogram = counter.create_Histogram(ch_trig=ch_trig, 
                                     ch_click=ch_click, 
                                     binwidth=bin_ns*1e3, 
                                     n_bins=int(1*seq_length/bin_ns))
start_index = 410
# index = [start_index//bin_ns + data['raman_cooling']/bin_ns + data['waiting_after_cooling']/bin_ns,
#         start_index//bin_ns + data['raman_cooling']/bin_ns + data['waiting_after_cooling']/bin_ns + data['rf_readout']/bin_ns,
#         start_index//bin_ns + data['raman_cooling']/bin_ns + data['waiting_after_cooling']/bin_ns + data['rf_readout']/bin_ns + data['gap_length'][0]/bin_ns,
#         start_index//bin_ns + data['raman_cooling']/bin_ns + data['waiting_after_cooling']/bin_ns + 2*data['rf_readout']/bin_ns + data['gap_length'][0]/bin_ns]
start = 824 + initial_delay  + delay_mw  

off = 23
values = [start,  start+raman_cooling]
# values = [start,  start+raman_cooling]
values.append(values[-1] + buffer)
values.append(values[-1] + 2500)
values.append(values[-1] + buffer)
values.append(values[-1] + pulse_length_ns)
values.append(values[-1] + buffer)
values.append(values[-1] + 2500)
# create figure
fig1 = plt.figure(figsize= (25,10))
t0 = time.time()

while True:

    histogram.startFor(int(integration_time_s*1e12), clear=True)
    fig1.clear() # clear old figure
    time.sleep(integration_time_s)
    counts = np.asarray(histogram.getData())
    time_ns = np.arange(len(counts))*bin_ns
    total_shots = integration_time_s*1e9/seq_length
    plt.plot(time_ns/1e0, counts/total_shots, '-', color='navy', lw=2)
    plt.xlabel('Time (us)', fontsize=50)
    plt.ylabel(f'Counts/shot', fontsize=50)
    # for _, _val in enumerate(values):
    #     plt.axvline (x=_val/1e0, color='red')
    plt.yticks(fontsize=50)
    plt.xticks(fontsize=50)
    plt.grid(alpha=0.5)
    # plt.xlim(values[0]-500, values[0]+500)
    # plt.xlim(values[3]-500, values[5]+500)
    # plt.xlim(left=values[3], right=values[-2]+1000)
    # plt.xlim(750, 1000)
    # plt.xlim(13300, 13600)
    # plt.xlim(800, 1200)~
    # plt.xlim(5500, 6000)
    # plt.xlim(8.3*1e3,8.5*1e3)
    # plt.axvline(x=13365, ls='--', color='red')
    # plt.axvline(x=13365 + pulse_length_ns , ls='--', color='red')
    # plt.axvline(x=13365  + 2*pulse_length_ns + ramsey_delay, ls='--', color='red')
    display(fig1)    
    clear_output(wait = True)
    histogram.clear()
    time.sleep(polling_interval)
    

KeyboardInterrupt: 

<Figure size 2500x1000 with 0 Axes>

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(time_ns/1e0, counts/total_shots, '-', color='navy')
plt.xlabel('Time (us)')
plt.ylabel(f'Counts/shot')
for _, _val in enumerate(values):
    plt.axvline (x=_val/1e0, ls='--'
                 ,color='red')
# plt.yticks(fontsize=30)
# plt.xticks(fontsize=30)
plt.grid(alpha=0.5)
# plt.xlim(values[1]-500, values[3]+500)

# External Trigger Pins

![External trigger pins](../images/external_trigger_pins.jpg)

In [12]:
soc.reset_gens()